In [4]:
# Cell 1 - Import libraries
import spacy
import warnings
warnings.filterwarnings('ignore')

# Load english model
nlp = spacy.load('en_core_web_sm')

print("spaCy loaded successfully!")

spaCy loaded successfully!


In [ ]:
# Cell 2 - Severity keyword configuration
# Replace these lists with your own keywords for each severity level.
severity_keyword_rules = {
    'no_injury': [
        "no injury",
        "no injuries",
        "no one injured",
        "no one is injured",
        "not injured",
        "without injury",
        "unhurt",
        "no casualties",
        "no one hurt",
        "no blood",
        "safe",
        "okay",
        "all okay",
        "safe only",
        "nothing serious",
    ],
    'high': [
        "dead",
        "died",
        "death",
        "unconscious",
        "not breathing",
        "no breathing",
        "can't breathe",
        "cant breathe",
        "heavy bleeding",
        "blood everywhere",
        "bleeding badly",
        "trapped",
        "stuck inside",
        "crushed",
        "head injury",
        "skull fracture",
        "broken neck",
        "spine injury",
        "chest injury",
        "severe pain",
        "leg cut off",
        "hand cut off",
        "multiple fractures",
        "body not moving",
        "heart attack",
        "seizure",
        "fire",
        "vehicle burning",
        "explosion",
        "smoke from car",
        "car upside down",
        "rollover",
        "fell from bridge",
        "fell in ditch",
        "hit by lorry",
        "truck crash",
        "bus accident",
        "high speed crash",
        "major accident",
        "fatal accident",
        "person under vehicle",
        "pinned under bike",
        "emergency",
        "help immediately",
        "save me",
        "critical",
        "serious condition",
        # Broken English / Chennai Style
        "lot blood coming",
        "person not waking up",
        "body stuck",
        "bike fully damage",
        "car burning",
        "head break",
        "leg break badly",
        "very big accident",
        "cannot move",
        "heavy injury",
        "life danger",
        "full blood",
        "person under vehicle",
    ],
    'medium': [
        "fracture",
        "broken arm",
        "broken leg",
        "injured",
        "pain in chest",
        "pain in back",
        "neck pain",
        "dizziness",
        "fainted",
        "swelling",
        "deep cut",
        "bleeding",
        "shoulder injury",
        "hand injury",
        "leg injury",
        "unable to stand",
        "unable to walk",
        "vehicle hit divider",
        "collision",
        "side impact",
        "rear end collision",
        "bike skid",
        "bike slipped",
        "hit by car",
        "hit by bike",
        "moderate damage",
        "airbag deployed",
        "passenger injured",
        "driver injured",
        # Broken English / Chennai Style
        "hand broken",
        "leg broken",
        "too much pain",
        "can't walk",
        "cant walk",
        "can't stand",
        "cant stand",
        "bike skidded",
        "bike slipped",
        "car hit",
        "fell down road",
        "injury in hand",
        "injury in leg",
        "body pain",
        "feeling dizzy",
        "little bleeding",
        "shoulder pain",
    ],
    'low': [
        "scratch",
        "small scratch",
        "minor injury",
        "no injury",
        "safe",
        "okay",
        "slight pain",
        "little pain",
        "bruises",
        "small cut",
        "vehicle damage",
        "bumper damage",
        "mirror broken",
        "indicator broken",
        "slow speed accident",
        "parking accident",
        "tyre burst",
        "skid but safe",
        "fell but okay",
        "no bleeding",
        "conscious",
        "alert",
        # Broken English / Chennai Style
        "small accident",
        "little damage",
        "bike scratch",
        "car scratch",
        "nothing serious",
        "all okay",
        "minor hit",
        "just fell",
        "safe only",
        "little pain",
        "vehicle okay",
        "manageable",
    ],
}

print('Severity keyword rules loaded with user-provided lists.')

Severity keyword rules loaded.


In [6]:
# Cell 3 - Extract accident info from text
import re


def _matches_any(patterns, text):
    return any(re.search(pattern, text) for pattern in patterns)


def extract_accident_info(text):
    doc = nlp(text)
    text_lower = text.lower()

    no_injury_patterns = [re.escape(keyword) for keyword in severity_keyword_rules.get('no_injury', [])]
    high_patterns = [re.escape(keyword) for keyword in severity_keyword_rules.get('high', [])]
    medium_patterns = [re.escape(keyword) for keyword in severity_keyword_rules.get('medium', [])]
    low_patterns = [re.escape(keyword) for keyword in severity_keyword_rules.get('low', [])]

    if _matches_any(no_injury_patterns, text_lower):
        severity = 'Low'
    else:
        score = 0
        if _matches_any(high_patterns, text_lower):
            score += 3
        if _matches_any(medium_patterns, text_lower):
            score += 2
        if _matches_any(low_patterns, text_lower):
            score -= 1

        if score >= 3:
            severity = 'High'
        elif score >= 2:
            severity = 'Medium'
        else:
            severity = 'Low'

    # Extract locations using spaCy
    locations = [ent.text for ent in doc.ents if ent.label_ in ['GPE', 'LOC', 'FAC']]

    # Extract numbers
    numbers = [ent.text for ent in doc.ents if ent.label_ == 'CARDINAL']

    return {
        'severity': severity,
        'locations': locations,
        'people_count': numbers,
        'original_text': text
    }


# Test it
examples = [
    'No injury, just a small scratch after the crash',
    'Bad accident on Anna Salai near Gemini flyover, 3 people injured, one is unconscious',
    'Two people injured in a bike accident',
]
for sample in examples:
    result = extract_accident_info(sample)
    print('Input:', result['original_text'])
    print('Severity:', result['severity'])
    print('Locations found:', result['locations'])
    print('Numbers found:', result['people_count'])
    print('-')

Input: No injury, just a small scratch after the crash
Severity: Low
Locations found: []
Numbers found: []
-
Input: Bad accident on Anna Salai near Gemini flyover, 3 people injured, one is unconscious
Severity: High
Locations found: ['Gemini']
Numbers found: ['3']
-
Input: Two people injured in a bike accident
Severity: Medium
Locations found: []
Numbers found: ['Two']
-


In [2]:
# Cell 3 - Voice input using Whisper
import whisper
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wav

# Load whisper model (tiny = fastest, good enough for us)
print("Loading Whisper model...")
whisper_model = whisper.load_model("tiny")
print("Whisper loaded!")

def record_audio(duration=5, sample_rate=16000):
    print(f"🎤 Recording for {duration} seconds... SPEAK NOW!")
    audio = sd.rec(int(duration * sample_rate), 
                   samplerate=sample_rate, 
                   channels=1, dtype='float32')
    sd.wait()
    print("✅ Recording done!")
    return audio, sample_rate

def transcribe_audio(audio, sample_rate):
    # Save temporarily
    wav.write('temp_audio.wav', sample_rate, audio)
    # Transcribe
    result = whisper_model.transcribe('temp_audio.wav')
    return result['text']

print("Voice system ready!")

Loading Whisper model...


100%|█████████████████████████████████████| 72.1M/72.1M [00:13<00:00, 5.52MiB/s]


Whisper loaded!
Voice system ready!


In [6]:
# Cell 4 - Test voice recording and transcription
print("Testing voice input...")
print("You will have 5 seconds to describe an accident")
print("Say something like: 'Accident on Anna Salai, 2 people injured'")
print()

# Record audio
audio, sr = record_audio(duration=5)

# Transcribe
text = transcribe_audio(audio, sr)
print("📝 You said:", text)

# Extract info
result = extract_accident_info(text)
print("\n🚨 Extracted Info:")
print("Severity:", result['severity'])
print("Locations:", result['locations'])
print("People count:", result['people_count'])

Testing voice input...
You will have 5 seconds to describe an accident
Say something like: 'Accident on Anna Salai, 2 people injured'

🎤 Recording for 5 seconds... SPEAK NOW!
✅ Recording done!
📝 You said:  uncertainty again another three people injured one is unconscious

🚨 Extracted Info:
Severity: High
Locations: []
People count: ['three', 'one']
